# Figure 5D: per-solute explicit-correction RMSE across the test set

Per-solute bootstrap RMSE (¹H, solvent-averaged) for the explicit-solvent correction, ordered delta-22 -> olefin/pyridine isomers -> natural products, with a dashed "scaled to solute" baseline and a red "scaled to test set" line.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/applications", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
from applications_reader import Applications
import applications
import applications_plots
import paths

In [ ]:
APPLICATIONS_HDF5 = paths.dataset_file("applications", root=REPO)
XLSX = os.path.join(REPO, "data", "applications", "applications_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
loader = Applications(APPLICATIONS_HDF5, XLSX)
query_df_nn = applications.build_query_df_nn(loader)
seed = applications.build_bootstrap_seed_coeffs(loader)

In [ ]:
site_counts = (query_df_nn.drop_duplicates(subset=["solute", "nucleus", "site"])
               .groupby(["solute", "nucleus"]).size().unstack(fill_value=0))
per_solute = applications.per_solute_fits(query_df_nn)
all_solute = applications.per_solvent_fits(query_df_nn)
preds_h = applications.apply_bootstrap_params_to_full_dataset(query_df_nn, seed["H"], nucleus="H")
bootstrap_rmses_h = applications.compute_solute_rmses(preds_h)

In [ ]:
FIG5D_LABELS = {
    "isomer_1E": "Olefin 1 (E)", "isomer_1Z": "Olefin 1 (Z)",
    "isomer_2E": "Olefin 2 (E)", "isomer_2Z": "Olefin 2 (Z)",
    "isomer_3E": "Olefin 3 (E)", "isomer_3Z": "Olefin 3 (Z)",
    "isomer_4N": "Pyridone 4", "isomer_4O": "Pyridine 4",
    "vomicine": "Vomicine", "prednisone": "Prednisone",
    "peptide": "Acetyl-L-alanyl-L-\nglutamine", "flavone": "Flavone",
    "dihydrotanshinone_I": "Dihydrotanshinone I",
}
FIG5D_ORDER = ["isomer_1E", "isomer_1Z", "isomer_2E", "isomer_2Z", "isomer_3E", "isomer_3Z",
               "isomer_4N", "isomer_4O", "vomicine", "prednisone", "peptide", "flavone",
               "dihydrotanshinone_I"]

In [ ]:
applications_plots.plot_nps_on_boxplot_delta22_simplified(
    loader.rmse_distribution("H"), bootstrap_rmses_h, per_solute["H"],
    nucleus="H", formulas=["stationary_plus_qcd + openMM"], colors=["#61a89a"],
    site_counts=site_counts, formula_remap=applications.FORMULA_REMAP,
    solute_remap=FIG5D_LABELS, solute_order=FIG5D_ORDER,
    title="Complex Molecules are Dominated by Additional Physics",
    solute_color_remap=applications.PEPTIDE_HIGHLIGHT_H,
    figsize=(14, 8), box_width=0.20, box_gap=0.1,
    show_baseline=True, baseline_annotation_text="Scaled to solute",
    baseline_annotation_x=0.215, baseline_annotation_y=0.39,
    show_full_fit_line=True, full_fit_line_label="Scaled to Test Set", full_fit_line_label_x=0.835,
    all_solute_fitting_results=all_solute,
    max_bar_height=0.06, site_count_axis_mode="inset", site_count_inset_area_frac=0.1,
    site_count_inset_axis_offset=-0.06,
    save_path=figure_path("fig5d_complex_1H.png"))